In [1]:
import findspark

In [2]:
findspark.init()
findspark.find()

'/opt/spark'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [4]:
spark = SparkSession.builder.appName("TaxiOperationDFApp") \
        .config("spark.dymanmicAllocation.enabled", "false") \
        .config("spark.sql.adaptive.enabled", "false") \
        .getOrCreate()

25/04/03 23:07:35 WARN Utils: Your hostname, luffy-Latitude-3400 resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlp0s20f3)
25/04/03 23:07:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/03 23:07:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
sc = spark.sparkContext

In [6]:
sc

<SparkContext master=local[*] appName=TaxiOperationDFApp>

In [7]:
from IPython.display import *
display(HTML("<style>pre {white-space: pre !important;}</style>"))

In [8]:
employeesRdd = sc.parallelize([
                [1, "Neha", 1000],
                [2, "Steve", 2000],
                [3, "Kari", 3000],
                [4, "Ivan", 4000],
                [5, "Mohit", 5000]
])

In [9]:
employeesRdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:289

In [10]:
employeeDF = employeesRdd.toDF()
employeeDF.show()

+---+-----+----+
| _1|   _2|  _3|
+---+-----+----+
|  1| Neha|1000|
|  2|Steve|2000|
|  3| Kari|3000|
|  4| Ivan|4000|
|  5|Mohit|5000|
+---+-----+----+



In [11]:
employeeDF = employeesRdd.toDF(["Id", "Name", "Salary"])
employeeDF.show()

+---+-----+------+
| Id| Name|Salary|
+---+-----+------+
|  1| Neha|  1000|
|  2|Steve|  2000|
|  3| Kari|  3000|
|  4| Ivan|  4000|
|  5|Mohit|  5000|
+---+-----+------+



In [12]:
employeeDF.printSchema()

root
 |-- Id: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Salary: long (nullable = true)



In [13]:
yellowTaxiDF = spark.read \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .csv("/home/luffy/Documents/Spark/Data/YellowTaxis_202210.csv")

25/04/03 23:07:55 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [14]:
yellowTaxiDF.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-10-01 05:33:41|  2022-10-01 05:48:39|            1.0|          1.7|       1.0|                 N|         249|         107|           1|        9.5|  3.0|    0.5|      2.6

In [15]:
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [16]:
greemTaxiDF = spark.read \
                .option("header", "true") \
                .option("delimiter", "\t") \
                .csv("/home/luffy/Documents/Spark/Data/GreenTaxis_*.csv")

In [17]:
greemTaxiDF.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorId|lpep_pickup_datetime|lpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2|2022-10-01T06:08:...| 2022-10-01T06:21:...|            1.0|         2.47|       1.0|                 N|         256|         225|         1.0|       11.5|  0.5|    0.5|      2.5

In [18]:
greemTaxiDF.printSchema()

root
 |-- VendorId: string (nullable = true)
 |-- lpep_pickup_datetime: string (nullable = true)
 |-- lpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- RatecodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: string (nullable = true)
 |-- DOLocationID: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- congestion_surcharge: string (nullable = true)
 |-- airport_fee: string (nullable = true)



In [19]:
paymentTypesDF = spark.read.json("/home/luffy/Documents/Spark/Data/PaymentTypes.json")

In [20]:
paymentTypesDF.show()

+-----------+-------------+
|PaymentType|PaymentTypeID|
+-----------+-------------+
|Credit Card|            1|
|       Cash|            2|
|  No Charge|            3|
|    Dispute|            4|
|    Unknown|            5|
|Voided Trip|            6|
+-----------+-------------+



In [21]:
paymentTypesDF.printSchema()

root
 |-- PaymentType: string (nullable = true)
 |-- PaymentTypeID: long (nullable = true)



In [22]:
taxiBaseDF = spark.read \
                .option("multiline", "true") \
                .json("/home/luffy/Documents/Spark/Data/TaxiBases.json")

In [23]:
taxiBaseDF.show(truncate=False)

+-----------------------------------------------------------+----------+--------------------------------------+------------------------------------------------+--------------+------------+----------------+--------+---------------------------+
|Address                                                    |Date      |Entity Name                           |GeoLocation                                     |License Number|SHL Endorsed|Telephone Number|Time    |Type of Base               |
+-----------------------------------------------------------+----------+--------------------------------------+------------------------------------------------+--------------+------------+----------------+--------+---------------------------+
|{636, NEW YORK, 10001, NY, WEST   28 STREET}               |08/15/2019|VIER-NY,LLC                           |{40.75273, (40.75273, -74.006408), -74.006408}  |B02865        |No          |6466657536      |18:03:31|BLACK CAR BASE             |
|{131, BRONX, 10468, NY, KIN

In [24]:
taxiBaseDF.printSchema()

root
 |-- Address: struct (nullable = true)
 |    |-- Building: string (nullable = true)
 |    |-- City: string (nullable = true)
 |    |-- Postcode: long (nullable = true)
 |    |-- State: string (nullable = true)
 |    |-- Street: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Entity Name: string (nullable = true)
 |-- GeoLocation: struct (nullable = true)
 |    |-- Latitude: double (nullable = true)
 |    |-- Location: string (nullable = true)
 |    |-- Longitude: double (nullable = true)
 |-- License Number: string (nullable = true)
 |-- SHL Endorsed: string (nullable = true)
 |-- Telephone Number: long (nullable = true)
 |-- Time: string (nullable = true)
 |-- Type of Base: string (nullable = true)



In [25]:
yellowTaxiDF.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2022-10-01 05:33:41|  2022-10-01 05:48:39|            1.0|          1.7|       1.0|                 N|         249|         107|           1|        9.5|  3.0|    0.5|      2.6

In [26]:
yellowTaxiAnalyzedDF = yellowTaxiDF.describe("passenger_count", "trip_distance")

In [27]:
yellowTaxiAnalyzedDF.show()

+-------+------------------+-----------------+
|summary|   passenger_count|    trip_distance|
+-------+------------------+-----------------+
|  count|           3542392|          3675412|
|   mean|1.3846934500755421|6.206976298167459|
| stddev|0.9302303297407295|640.8236808320227|
|    min|               0.0|              0.0|
|    max|               9.0|        389678.46|
+-------+------------------+-----------------+



In [28]:
yellowTaxiDF = yellowTaxiDF.where("passenger_count>0") \
                .filter(col("trip_distance") > 0.0)
yellowTaxiDF.count()

3422296

In [29]:
yellowTaxiDF = yellowTaxiDF.na.drop('all')
yellowTaxiDF.count()

3422296

In [30]:
defaultValueMap = {"payment_type": 5, 'RateCodeID': 1}
yellowTaxiDF = yellowTaxiDF.na.fill(defaultValueMap)

In [31]:
yellowTaxiDF = yellowTaxiDF.dropDuplicates()
yellowTaxiDF.count()

25/04/03 23:08:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:35 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:36 WARN RowBasedKeyValueBatch: Calling spill() on

3422295

In [32]:
yellowTaxiDF = yellowTaxiDF.where("tpep_pickup_datetime >= '2022-10-01' AND tpep_dropoff_datetime < '2022-11-01'")
yellowTaxiDF.count()

25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:08:59 WARN RowBasedKeyValueBatch: Calling spill() on

3393897

In [33]:
yellowTaxiDF = yellowTaxiDF.select("VendorID",
                                    col("passenger_count").cast(IntegerType()),
                                    column("trip_distance").alias("TripDistance"),
                                    yellowTaxiDF.tpep_pickup_datetime,
                                    "tpep_dropoff_datetime",
                                    "PUlocationID",
                                    "DOlocationID",
                                    "RatecodeID",
                                    "total_amount",
                                    "payment_type"
                                  )
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PUlocationID: integer (nullable = true)
 |-- DOlocationID: integer (nullable = true)
 |-- RatecodeID: double (nullable = false)
 |-- total_amount: double (nullable = true)
 |-- payment_type: integer (nullable = false)



In [44]:
yellowTaxiDF = yellowTaxiDF \
                .withColumnRenamed("passenger_count", "PassengerCount") \
                .withColumnRenamed("tpep_pickup_datetime", "PickupTime") \
                .withColumnRenamed("tpep_dropoff_datetime", "DropTime") \
                .withColumnRenamed("PUlocationID", "PickupLocationId") \
                .withColumnRenamed("DOlocationID", "DropLocationId") \
                .withColumnRenamed("total_amount", "TotalAmount") \
                .withColumnRenamed("payment_type", "PaymentType")

yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- PassengerCount: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- PickupTime: timestamp (nullable = true)
 |-- DropTime: timestamp (nullable = true)
 |-- PickupLocationId: integer (nullable = true)
 |-- DropLocationId: integer (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentType: integer (nullable = false)
 |-- TripYear: integer (nullable = true)
 |-- TripMonth: integer (nullable = true)
 |-- TripDay: integer (nullable = true)
 |-- TripTimeInMinutes: double (nullable = true)
 |-- TripType: string (nullable = false)



In [37]:
yellowTaxiDF = yellowTaxiDF.withColumn("TripYear", year(col("PickupTime"))) \
                            .select("*", 
                                    expr("month(PickupTime) AS TripMonth"),
                                    dayofmonth(col("PickupTime")).alias("TripDay")
                                   )

In [38]:
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- PassengerCount: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- PickupTime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PickupLocationId: integer (nullable = true)
 |-- DropLocationId: integer (nullable = true)
 |-- RatecodeID: double (nullable = false)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentType: integer (nullable = false)
 |-- TripYear: integer (nullable = true)
 |-- TripMonth: integer (nullable = true)
 |-- TripDay: integer (nullable = true)



In [46]:
tripTimeInSecondsExpr = unix_timestamp(col("DropTime")) - unix_timestamp(col("PickupTime"))
tripTimeInMinutesExpr = round(tripTimeInSecondsExpr / 60)
yellowTaxiDF = yellowTaxiDF.withColumn("TripTimeInMinutes", tripTimeInMinutesExpr)
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- PassengerCount: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- PickupTime: timestamp (nullable = true)
 |-- DropTime: timestamp (nullable = true)
 |-- PickupLocationId: integer (nullable = true)
 |-- DropLocationId: integer (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentType: integer (nullable = false)
 |-- TripYear: integer (nullable = true)
 |-- TripMonth: integer (nullable = true)
 |-- TripDay: integer (nullable = true)
 |-- TripTimeInMinutes: double (nullable = true)
 |-- TripType: string (nullable = false)



In [41]:
tripTypeColumn = when(col("RatecodeID") == 6, "sharedTrip") \
                    .otherwise("SoloTrip")

yellowTaxiDF = yellowTaxiDF.withColumn("TripType", tripTypeColumn).drop("RatecodeID")
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- PassengerCount: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- PickupTime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- PickupLocationId: integer (nullable = true)
 |-- DropLocationId: integer (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentType: integer (nullable = false)
 |-- TripYear: integer (nullable = true)
 |-- TripMonth: integer (nullable = true)
 |-- TripDay: integer (nullable = true)
 |-- TripTimeInMinutes: double (nullable = true)
 |-- TripType: string (nullable = false)



In [42]:
yellowTaxiDF.explain(mode="extended")

== Parsed Logical Plan ==
Project [VendorID#55, PassengerCount#888, TripDistance#876, PickupTime#899, tpep_dropoff_datetime#57, PickupLocationId#920, DropLocationId#931, TotalAmount#942, PaymentType#953, TripYear#978, TripMonth#990, TripDay#991, TripTimeInMinutes#1006, TripType#1021]
+- Project [VendorID#55, PassengerCount#888, TripDistance#876, PickupTime#899, tpep_dropoff_datetime#57, PickupLocationId#920, DropLocationId#931, RatecodeID#769, TotalAmount#942, PaymentType#953, TripYear#978, TripMonth#990, TripDay#991, TripTimeInMinutes#1006, CASE WHEN (RatecodeID#769 = cast(6 as double)) THEN sharedTrip ELSE SoloTrip END AS TripType#1021]
   +- Project [VendorID#55, PassengerCount#888, TripDistance#876, PickupTime#899, tpep_dropoff_datetime#57, PickupLocationId#920, DropLocationId#931, RatecodeID#769, TotalAmount#942, PaymentType#953, TripYear#978, TripMonth#990, TripDay#991, round((cast((unix_timestamp(tpep_dropoff_datetime#57, yyyy-MM-dd HH:mm:ss, Some(Asia/Kolkata), false) - unix_ti

In [45]:
yellowTaxiDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- PassengerCount: integer (nullable = true)
 |-- TripDistance: double (nullable = true)
 |-- PickupTime: timestamp (nullable = true)
 |-- DropTime: timestamp (nullable = true)
 |-- PickupLocationId: integer (nullable = true)
 |-- DropLocationId: integer (nullable = true)
 |-- TotalAmount: double (nullable = true)
 |-- PaymentType: integer (nullable = false)
 |-- TripYear: integer (nullable = true)
 |-- TripMonth: integer (nullable = true)
 |-- TripDay: integer (nullable = true)
 |-- TripTimeInMinutes: double (nullable = true)
 |-- TripType: string (nullable = false)



In [47]:
yellowTaxiDFReport = yellowTaxiDF.groupBy("PickupLocationId", "DropLocationId") \
                        .agg(avg("TripTimeInMinutes").alias("AvgTripTime"), 
                             sum("TotalAmount").alias("SumAmount")
                            ) \
                        .orderBy(col("PickupLocationId").desc())

In [48]:
yellowTaxiDFReport.show()

25/04/03 23:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:22 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:23 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:28:23 WARN RowBasedKeyValueBatch: Calling spill() on

+----------------+--------------+------------------+------------------+
|PickupLocationId|DropLocationId|       AvgTripTime|         SumAmount|
+----------------+--------------+------------------+------------------+
|             265|            68|              26.0|            244.51|
|             265|           230|              37.0|            383.34|
|             265|           170|              40.0|350.43999999999994|
|             265|           116|              40.0|             53.16|
|             265|           218|              23.0|              45.0|
|             265|            86|23.666666666666668|              68.8|
|             265|           233|              45.0|             78.85|
|             265|           231|              10.0|              15.3|
|             265|           132|              31.9|            598.71|
|             265|            23|               2.0|              41.6|
|             265|            10|              11.0|            

In [51]:
yellowTaxiDF.write \
            .option("header", "true") \
            .option("dateFormat", "yyyy-MM-dd HH:mm:ss.S") \
            .mode("overwrite") \
            .csv("/home/luffy/Documents/Spark/Data/Output/YellowTaxisOutput")

25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
25/04/03 23:48:51 WARN RowBasedKeyValueBatch: Calling spill() on